In [4]:
import sys
import pandas as pd

# Permet d'importer validation_structure depuis le dossier starter-kit
sys.path.append("../starter-kit")
from validation_structure import normaliser, valider, PAYS_COUVERTS

df = pd.read_csv("../starter-kit/data/numeros_tva.csv", keep_default_na=False)

# Application directe de normaliser() et valider()
df["tva_norm"] = df["numero_tva"].apply(normaliser)
resultats = df["numero_tva"].apply(valider)
df["valide_struct"] = resultats.apply(lambda x: x[0])
df["motif_brut"] = resultats.apply(lambda x: x[1])

print("Répartition des motifs bruts :")
print(df["motif_brut"].value_counts())

Répartition des motifs bruts :
motif_brut
ok                  6083
cle_invalide        2605
pays_non_couvert    1090
vide                 174
trop_court            48
Name: count, dtype: int64


In [2]:
# 1. Que contient réellement "trop_court" ?
print("=== Détail de trop_court ===")
print(df[df["motif_brut"] == "trop_court"][["numero_tva", "tva_norm"]].value_counts())

# 2. Que contient réellement "pays_non_couvert" ?
pnc = df[df["motif_brut"] == "pays_non_couvert"]
print("\n=== Top 15 des 2 premiers caractères dans pays_non_couvert ===")
print(pnc["tva_norm"].str[:2].value_counts().head(15))

# 3. Combien de lignes dans pays_non_couvert commencent par un chiffre (préfixe pays oublié) ?
sans_prefixe = pnc[pnc["tva_norm"].str[0].str.isdigit()]
print("\nNuméros sans préfixe pays au début :", len(sans_prefixe))
print(sans_prefixe[["id", "pays_declare", "numero_tva", "tva_norm"]].head(10))

=== Détail de trop_court ===
numero_tva  tva_norm
N/A         NA          48
Name: count, dtype: int64

=== Top 15 des 2 premiers caractères dans pays_non_couvert ===
tva_norm
ZZ    115
QQ    109
GB    104
UK    104
XX     87
NU     39
25     13
19     12
38     11
16     11
02     10
42     10
14     10
95     10
58      9
Name: count, dtype: int64

Numéros sans préfixe pays au début : 532
      id pays_declare    numero_tva      tva_norm
119  120           FR   58560322811   58560322811
134  135           NL  396904956B68  396904956B68
162  163           IT   38486825359   38486825359
180  181           IT   33139885439   33139885439
188  189           IT   88838354509   88838354509
194  195           SE  107327005801  107327005801
246  247           PL    5112134613    5112134613
259  260           LU      75041556      75041556
261  262           BE    0572894866    0572894866
263  264           IT   09184526433   09184526433


In [3]:
def nettoyer_et_valider(row):
    norm = normaliser(row["numero_tva"])
    
    # 1. Rattraper les faux textes vides (N/A -> NA, null / NU.LL -> NULL)
    if norm in ("", "NA", "NULL"):
        return "", "vide"
    
    # 2. Reconstruire le préfixe pays si le numéro commence par un chiffre
    if norm[0].isdigit():
        norm = row["pays_declare"].strip().upper() + norm
        
    # 3. Appliquer le module de validation structurelle
    valide, motif = valider(norm)
    return norm, motif

res = df.apply(nettoyer_et_valider, axis=1)
df["tva_propre"] = res.apply(lambda x: x[0])
df["motif_propre"] = res.apply(lambda x: x[1])

print("Nouvelle répartition des motifs :")
print(df["motif_propre"].value_counts())

print("\nDétail des pays restants dans pays_non_couvert :")
print(df[df["motif_propre"] == "pays_non_couvert"]["tva_propre"].str[:2].value_counts())

Nouvelle répartition des motifs :
motif_propre
ok                  6615
cle_invalide        2605
pays_non_couvert     519
vide                 261
Name: count, dtype: int64

Détail des pays restants dans pays_non_couvert :
tva_propre
ZZ    115
QQ    109
GB    104
UK    104
XX     87
Name: count, dtype: int64
